# Deep Neural Networks

Ya construimos y entrenamos nuestra primera red neuronal.

Ahora daremos el siguiente paso:

> **¿Qué cambia cuando añadimos más capas ocultas?**

Una **Deep Neural Network (DNN)** es una red neuronal con múltiples hidden layers.

Por ejemplo:

```text
Red sencilla:
4 → 8 → 3

Deep Neural Network:
4 → 32 → 16 → 8 → 3
```

La idea no es añadir capas porque sí.

Cada capa puede aprender representaciones intermedias cada vez más complejas.

## Objetivos

Al finalizar podrás:

- explicar qué hace "deep" a una red;
- interpretar arquitecturas profundas;
- calcular parámetros en una DNN;
- construir una DNN con PyTorch;
- comparar una red sencilla con una profunda;
- observar cómo la capacidad del modelo cambia con la arquitectura;
- entender por qué más profundidad no siempre significa mejor desempeño.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn


## 1. De una red sencilla a una red profunda

Una red con una hidden layer podría ser:

```text
INPUT       HIDDEN       OUTPUT

13  ───────→ 16 ───────→ 3
```

Una red más profunda:

```text
INPUT       HIDDEN       HIDDEN       HIDDEN       OUTPUT

13  ───────→ 32 ───────→ 16 ───────→ 8 ───────→ 3
```

La segunda red contiene más transformaciones:

\[
x
\rightarrow
h_1
\rightarrow
h_2
\rightarrow
h_3
\rightarrow
\hat{y}.
\]


## 2. ¿Qué puede aprender cada capa?

Podemos pensar conceptualmente:

```text
Raw features
    ↓
Layer 1
    ↓
combinaciones simples
    ↓
Layer 2
    ↓
patrones intermedios
    ↓
Layer 3
    ↓
representaciones más complejas
    ↓
Output
```

En datos científicos esto puede significar que la red aprende combinaciones útiles de mediciones originales.

No significa necesariamente que podamos interpretar cada neurona como una propiedad física específica.


## 3. Dataset

Usaremos el dataset Wine como un conjunto de mediciones químicas multivariadas.

Tiene:

```text
13 features
3 clases
```


In [ ]:
torch.manual_seed(42)

wine = load_wine()

X = wine.data
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

print("Training:", X_train_t.shape)
print("Test:", X_test_t.shape)


## 4. Red sencilla

Construiremos primero:

```text
13 → 16 → 3
```


In [ ]:
simple_model = nn.Sequential(
    nn.Linear(13, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

print(simple_model)


## 5. Red profunda

Ahora:

```text
13 → 32 → 16 → 8 → 3
```


In [ ]:
deep_model = nn.Sequential(
    nn.Linear(13, 32),
    nn.ReLU(),

    nn.Linear(32, 16),
    nn.ReLU(),

    nn.Linear(16, 8),
    nn.ReLU(),

    nn.Linear(8, 3)
)

print(deep_model)


## 6. Cuenta los parámetros

Antes de ejecutar, calcula cuántos parámetros tiene la red profunda.

<details>
<summary><strong>Pista</strong></summary>

Calcula capa por capa:

\[
13\to32
\]

\[
32\to16
\]

\[
16\to8
\]

\[
8\to3.
\]

Para cada capa:

\[
n_{\text{params}}
=
n_{\text{in}}n_{\text{out}}
+
n_{\text{out}}.
\]

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

Primera capa:

\[
13\times32+32=448
\]

Segunda:

\[
32\times16+16=528
\]

Tercera:

\[
16\times8+8=136
\]

Salida:

\[
8\times3+3=27
\]

Total:

\[
448+528+136+27=1139
\]

parámetros.

</details>


In [ ]:
simple_params = sum(p.numel() for p in simple_model.parameters())
deep_params = sum(p.numel() for p in deep_model.parameters())

print("Simple model parameters:", simple_params)
print("Deep model parameters:  ", deep_params)


## 7. Función de entrenamiento

Crearemos una función para entrenar ambos modelos de la misma manera.


In [ ]:
def train_model(model, X, y, epochs=300, lr=0.05):

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    history = []

    for epoch in range(epochs):

        logits = model(X)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history.append(loss.item())

    return history


## 8. Entrenamos ambos modelos

Para una comparación justa, reiniciamos la semilla antes de construir cada uno.


In [ ]:
torch.manual_seed(42)

simple_model = nn.Sequential(
    nn.Linear(13, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

simple_history = train_model(
    simple_model,
    X_train_t,
    y_train_t,
    epochs=300,
    lr=0.05
)

torch.manual_seed(42)

deep_model = nn.Sequential(
    nn.Linear(13, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

deep_history = train_model(
    deep_model,
    X_train_t,
    y_train_t,
    epochs=300,
    lr=0.05
)


## 9. Comparamos la training loss


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(simple_history, label="Red sencilla")
plt.plot(deep_history, label="DNN")
plt.xlabel("Epoch")
plt.ylabel("Cross Entropy Loss")
plt.title("Training Loss")
plt.legend()
plt.show()


### Pregunta

¿La DNN necesariamente tiene una loss menor desde el inicio?

<details>
<summary><strong>Pista</strong></summary>

Más parámetros también significa un espacio de optimización más complejo.

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

No necesariamente.

Una red más profunda puede tener mayor capacidad, pero puede requerir:

- más epochs;
- un learning rate diferente;
- otro optimizer;
- regularización;
- mejor inicialización.

Más profundidad no garantiza automáticamente un entrenamiento mejor.

</details>


## 10. Accuracy

Definamos una función:


In [ ]:
def accuracy(model, X, y):
    with torch.no_grad():
        logits = model(X)
        predictions = torch.argmax(logits, dim=1)
        return (predictions == y).float().mean().item()


In [ ]:
print("Simple model:")
print("  train =", accuracy(simple_model, X_train_t, y_train_t))
print("  test  =", accuracy(simple_model, X_test_t, y_test_t))

print("\nDeep model:")
print("  train =", accuracy(deep_model, X_train_t, y_train_t))
print("  test  =", accuracy(deep_model, X_test_t, y_test_t))


## 11. Capacidad del modelo

Una red más profunda tiene más parámetros y, por lo tanto, mayor **capacity**.

Eso puede ser útil para aprender relaciones complejas.

Pero también aumenta el riesgo de:

```text
memorizar demasiado el training set
          ↓
poor generalization
```

A esto lo llamamos **overfitting**.


## 12. Reto

Construye una arquitectura:

```text
13 → 64 → 32 → 16 → 3
```

y calcula:

1. número de hidden layers;
2. número total de parámetros.

### Tu código


In [ ]:
# Escribe tu modelo aquí



<details>
<summary><strong>Pista</strong></summary>

Usa `nn.Sequential`, `nn.Linear` y `nn.ReLU`.

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

```python
model_large = nn.Sequential(
    nn.Linear(13, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

print(sum(p.numel() for p in model_large.parameters()))
```

Tiene **3 hidden layers**.

</details>


# Para recordar

Una DNN puede representarse como:

```text
Input
  ↓
Hidden Layer
  ↓
Hidden Layer
  ↓
Hidden Layer
  ↓
Output
```

Más profundidad aumenta la capacidad del modelo.

Pero también debemos preocuparnos por:

- overfitting;
- estabilidad del entrenamiento;
- número de parámetros;
- datos disponibles;
- generalización.

Eso nos lleva al próximo tema:

# Overfitting y Regularización


## Recursos

- [PyTorch — Build the Neural Network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
- [Deep Learning Book — Deep Feedforward Networks](https://www.deeplearningbook.org/contents/mlp.html)
